<a href="https://colab.research.google.com/github/ochilovu2010/IOAI/blob/main/IOAI_2025/Restroom_Problem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

restroom_ioai_2025_path = kagglehub.competition_download('restroom-ioai-2025')

print('Data source import complete.')


# Restroom Icon Matching

In [ ]:
import random
import torch
from torch import nn
import os
from PIL import Image
import transformers
from transformers import CLIPVisionModel, CLIPModel
from torch.utils.data import Dataset, DataLoader

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model_name = 'openai/clip-vit-base-patch16'
imagemodel = CLIPVisionModel.from_pretrained(model_name)
gendermodel = CLIPVisionModel.from_pretrained(model_name)

processor = CLIPProcessor.from_pretrained(model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
vision_model.embeddings.position_ids                         | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
vision_model.embeddings.position_ids                         | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias 

In [ ]:
imagemodel.to(device)
gendermodel.to(device)

CLIPVisionModel(
  (vision_model): CLIPVisionTransformer(
    (embeddings): CLIPVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
      (position_embedding): Embedding(197, 768)
    )
    (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
       

In [ ]:
for params in imagemodel.parameters():
    params.requires_grad = False

for layers in imagemodel.vision_model.encoder.layers[-6:]:
    for p in layers.parameters():
        p.requires_grad = True

for params in gendermodel.parameters():
    params.requires_grad = False

for layers in gendermodel.vision_model.encoder.layers[-6:]:
    for p in layers.parameters():
        p.requires_grad = True



In [ ]:
total = 0
active = 0
for x in imagemodel.parameters():
    total+=1
    if x.requires_grad == True:
        active +=1
print(active/total)
print(active, total)

0.4824120603015075
96 199


In [ ]:
class crop2orig(Dataset):
    def __init__(self, processor):
        crop = "/kaggle/input/competitions/restroom-ioai-2025/training_set/training_set/crop"
        orig = "/kaggle/input/competitions/restroom-ioai-2025/training_set/training_set/orig"
        self.crop = []
        self.orig = []
        self.processor = processor
        for x in os.listdir(crop):
            for i in os.listdir(os.path.join(crop, x)):
                self.crop.append(os.path.join(crop, x, i))
                self.orig.append(os.path.join(orig, x, i))
    def __len__(self):
        return len(self.crop)
    def __getitem__(self,x):
        crop_image = self.processor(images = Image.open(self.crop[x]), return_tensors = 'pt')['pixel_values']
        orig_image = self.processor(images = Image.open(self.orig[x]), return_tensors = 'pt')['pixel_values']
        return {
            "crop_pixel": crop_image.squeeze(0),
            "orig_pixel": orig_image.squeeze(0)
        }


In [ ]:
class genderdataset(Dataset):
    def __init__(self, processor):
        male = "/kaggle/input/competitions/restroom-ioai-2025/training_set/training_set/orig/male"
        female = "/kaggle/input/competitions/restroom-ioai-2025/training_set/training_set/orig/female"
        self.female = []
        self.male = []
        self.processor = processor
        for x in os.listdir(male):
            self.male.append(os.path.join(male, x))
            self.female.append(os.path.join(female, x))
    def __len__(self):
        return len(self.male)
    def __getitem__(self,x):
        male_image = self.processor(images = Image.open(self.male[x]), return_tensors = 'pt')['pixel_values']
        female_image = self.processor(images = Image.open(self.female[x]), return_tensors = 'pt')['pixel_values']
        return {
            "male": male_image.squeeze(0),
            "female": female_image.squeeze(0)
        }


In [ ]:
image = crop2orig(processor)
gender = genderdataset(processor)
image_loader = DataLoader(image, batch_size = 32)
gender_loader = DataLoader(gender, batch_size =32)

In [ ]:
from tqdm import tqdm
epochs = 10
import torch.nn.functional as F
optimizer = torch.optim.Adam(imagemodel.parameters(), lr = 1e-4)
for epoch in range(epochs):
    running_loss = []
    for batch in tqdm(image_loader):

        crop_outputs = imagemodel(pixel_values = batch['crop_pixel'].to(device)).pooler_output
        orig_outputs = imagemodel(pixel_values = batch['orig_pixel'].to(device)).pooler_output
        crop = F.normalize(crop_outputs, dim = 1)
        orig = F.normalize(orig_outputs, dim = 1)
        logits = crop @ orig.T
        temperature = 0.07
        logits = logits/temperature
        labels = torch.arange(logits.shape[0], device = device)
        loss1 = F.cross_entropy(logits, labels)
        loss2 = F.cross_entropy(logits.T, labels)
        loss = (loss1 + loss2) / 2
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss.append(loss.item())
    print(sum(running_loss)/len(running_loss))

100%|██████████| 6/6 [00:06<00:00,  1.04s/it]


0.2238864010820786


100%|██████████| 6/6 [00:06<00:00,  1.06s/it]


0.013811798085347013


100%|██████████| 6/6 [00:06<00:00,  1.09s/it]


...

0.00038831871794779244


100%|██████████| 6/6 [00:06<00:00,  1.12s/it]


0.0003284705141292458


100%|██████████| 6/6 [00:06<00:00,  1.04s/it]

0.0002877627690243874


In [ ]:
from tqdm import tqdm
epochs = 10
import torch.nn.functional as F
optimizer_gender = torch.optim.Adam(gendermodel.parameters(), lr = 1e-4)
for epoch in range(epochs):
    running_loss = []
    for batch in tqdm(gender_loader):

        male_outputs = gendermodel(pixel_values = batch['male'].to(device)).pooler_output
        female_outputs = gendermodel(pixel_values = batch['female'].to(device)).pooler_output
        male = F.normalize(male_outputs, dim = 1)
        female = F.normalize(female_outputs, dim = 1)
        logits = male @ female.T
        temperature = 0.07
        logits = logits/temperature
        labels = torch.arange(logits.shape[0], device = device)
        loss1 = F.cross_entropy(logits, labels)
        loss2 = F.cross_entropy(logits.T, labels)
        loss = (loss1 + loss2) / 2
        optimizer_gender.zero_grad()
        loss.backward()
        optimizer_gender.step()

        running_loss.append(loss.item())
    print(sum(running_loss)/len(running_loss))

100%|██████████| 3/3 [00:03<00:00,  1.06s/it]


0.39983369906743366


100%|██████████| 3/3 [00:03<00:00,  1.15s/it]


0.01801796133319537


100%|██████████| 3/3 [00:03<00:00,  1.17s/it]


...

0.0007615663732091585


100%|██████████| 3/3 [00:03<00:00,  1.06s/it]


0.0005949712455427895


100%|██████████| 3/3 [00:03<00:00,  1.08s/it]

0.0004836998802299301


In [ ]:
imagemodel.eval()
gendermodel.eval()


def sort_paths_by_number(paths):
    paths.sort(key=lambda x: int(os.path.splitext(os.path.basename(x))[0]))


def infer(model, image_paths):
    feats = []

    with torch.no_grad():
        for path in image_paths:
            image = Image.open(path).convert("RGB")

            pixel_values = processor(
                images=image,
                return_tensors="pt"
            )["pixel_values"].to(device)

            feat = model(pixel_values=pixel_values).pooler_output
            feat = F.normalize(feat, dim=1)

            feats.append(feat.cpu())

    return torch.cat(feats, dim=0)


def match_images(base_dir):

    query_dir = base_dir / "query"
    gallery_dir = base_dir / "gallery"

    query_paths = [str(x) for x in query_dir.glob("*.png")]
    gallery_paths = [str(x) for x in gallery_dir.glob("*.png")]

    sort_paths_by_number(query_paths)
    sort_paths_by_number(gallery_paths)


    query_feat = infer(imagemodel, query_paths)

    gallery_image_feat = infer(imagemodel, gallery_paths)

    gallery_gender_feat = infer(gendermodel, gallery_paths)

    sim = query_feat @ gallery_image_feat.T

    matched_gallery = sim.argmax(dim=1)

    final_predictions = []

    for idx in matched_gallery:

        idx = idx.item()

        feat = gallery_gender_feat[idx].unsqueeze(0)

        sim = (feat @ gallery_gender_feat.T).squeeze()

        sim[idx] = -1e9

        prediction = sim.argmax().item()

        final_predictions.append(prediction + 1)

    return final_predictions


DATA_PATH = Path("/kaggle/input/competitions/restroom-ioai-2025/test_set/")

print("Running inference...")

predictions = match_images(DATA_PATH / "test_set")

submission = pd.DataFrame({
    "id": [0],
    "array": [str(predictions)]
})

submission.to_csv("submission.csv", index=False)

print("submission.csv saved!")

print(submission.head())

Running inference...
submission.csv saved!
   id                                              array
0   0  [37, 13, 45, 17, 58, 49, 14, 59, 39, 7, 18, 21...


In [ ]:
submission

,id,array
0,0,"[37, 13, 45, 17, 58, 49, 14, 59, 39, 7, 18, 21..."
